In [ ]:
# Cell 1: 导入所需库
# ============================================================================
# 本项目涉及数据清洗、缺失值处理、特征工程、特征选择等步骤，
# 所需的工具库主要来自 pandas/numpy（数据处理）和 sklearn（预处理与特征选择）。

import pandas as pd
import numpy as np
import warnings
from sklearn.preprocessing import (StandardScaler, MinMaxScaler, LabelEncoder,
                                   OrdinalEncoder, OneHotEncoder, KBinsDiscretizer,
                                   Binarizer, FunctionTransformer)
from sklearn.impute import SimpleImputer
warnings.filterwarnings('ignore')

# 设置显示选项：避免因列数/行数过多导致输出被截断，方便检查数据
pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 50)
pd.set_option('display.float_format', '{:.4f}'.format)

In [ ]:
# Cell 2: 加载两个 xlsx 文件并合并
# ============================================================================
# 数据来源：两份医保理赔 Excel 文件，每份 108 列。
# 加载时统一用 dtype=str 避免 pandas 自动推断类型导致数据丢失（如前导零被吃掉）。
# 合并后去重，得到完整的原始数据集。

file1 = 'data-14-01.xlsx'
file2 = 'data-18-01.xlsx'

df1 = pd.read_excel(file1, dtype=str)
df2 = pd.read_excel(file2, dtype=str)

print(f"文件1形状: {df1.shape}，文件2形状: {df2.shape}")
df_raw = pd.concat([df1, df2], axis=0, ignore_index=True)
df_raw = df_raw.drop_duplicates()
print(f"合并后形状: {df_raw.shape}")
print("原始列名列表：")
print(df_raw.columns.tolist())

In [ ]:
# Cell 3: 字段筛选 – 保留核心列
# ============================================================================
# 原始数据 108 列，包含大量管理字段（付款人姓名、银行账号、系统内部ID等），
# 这些字段对欺诈检测没有意义，反而会增加噪声。
# 我们按业务逻辑分 8 大类，保留 47 个核心字段。
# 注意：RJ_CODE_LIST 和 CODES 保留是因为后续标签构造需要用到它们，但建模前会删除。

keep_cols = [
    # 理赔金额 (7)
    'ORG_PRES_AMT_VALUE', 'APP_AMT', 'BEN_SPEND', 'PAY_AMT_USD', 'SUB_AMT',
    'REJECTED_AMT', 'TOTAL_RECEIPT_AMT',
    # 付款结构 (6)
    'CL_SOCIAL_PAY_AMT', 'CL_THIRD_PARTY_PAY_AMT', 'CL_OWNER_PAY_AMT',
    'CL_SELF_CAT_PAY_AMT', 'DED_AMT', 'COPAY_PCT',
    # 诊断信息 (5)
    'DIAG_CODE', 'CODES', 'BEN_HEAD', 'BEN_HEAD_TYPE', 'SCMA_OID_BEN_TYPE',
    # 医院信息 (4)
    'PROV_CODE', 'PROV_LEVEL', 'PROV_DEPT', 'CLSH_HOSP_CODE',
    # 理赔状态 (3) —— RJ_CODE_LIST 保留用于标签生成，CODES 也保留但建模前删除
    'SCMA_OID_CL_LINE_STATUS', 'SCMA_OID_CL_STATUS', 'RJ_CODE_LIST',
    # 被保险人 (3)
    'MBR_NO', 'MBR_TYPE', 'NO_OF_YR',
    # 保单信息 (5)
    'POCY_NO', 'POCY_PLAN_DESC', 'PLAN_OID', 'POPL_OID', 'POHO_NO',
    # 时间特征 (5)
    'INCUR_DATE_FROM', 'INCUR_DATE_TO', 'PAY_DATE', 'RCV_DATE', 'FILE_CLOSE_DATE',
    # 行/单据信息 (2)
    'CL_NO', 'CL_LINE_NO',
    # 其他特征 (7)
    'KIND_CODE', 'INSUR_INVOICE_IND', 'INVOICE_CNT', 'FX_RATE',
    'POLICY_CNT', 'CWF_AMT_DAY', 'SCMA_OID_COUNTRY_TREATMENT'
]

# 统一列名为大写，确保后续匹配不出错
df_raw.columns = df_raw.columns.str.strip().str.upper()
keep_cols_upper = [col.upper() for col in keep_cols]
existing_cols = [col for col in keep_cols_upper if col in df_raw.columns]
missing_cols = set(keep_cols_upper) - set(existing_cols)
if missing_cols:
    print(f"警告：以下列在数据中不存在，将被忽略: {missing_cols}")
df = df_raw[existing_cols].copy()
print(f"筛选后数据形状: {df.shape}")

In [ ]:
# Cell 4: 金额列清洗
# ============================================================================
# 原始金额字段带有"RMB"前缀、逗号分隔符和空格，属于字符串格式，
# 无法直接进行数学运算。需要去掉这些干扰字符后转为数值类型。
# errors='coerce' 表示无法转换的值变为 NaN，避免程序报错。

def clean_currency(series):
    """去除金额字符串中的 RMB 前缀、逗号和空格"""
    if series.dtype == 'object':
        s = series.astype(str).str.replace('RMB', '', case=False, regex=False)
        s = s.str.replace(',', '', regex=False)
        s = s.str.replace(' ', '', regex=False)
        s = s.str.strip()
        return s
    return series

amt_cols = [
    'ORG_PRES_AMT_VALUE', 'APP_AMT', 'BEN_SPEND', 'PAY_AMT_USD', 'SUB_AMT',
    'REJECTED_AMT', 'TOTAL_RECEIPT_AMT',
    'CL_SOCIAL_PAY_AMT', 'CL_THIRD_PARTY_PAY_AMT', 'CL_OWNER_PAY_AMT',
    'CL_SELF_CAT_PAY_AMT', 'DED_AMT', 'CWF_AMT_DAY'
]
amt_cols = [col for col in amt_cols if col in df.columns]
for col in amt_cols:
    df[col] = clean_currency(df[col])
    df[col] = pd.to_numeric(df[col], errors='coerce')
print("金额列转换为数值完成")

In [ ]:
# Cell 5: 日期列处理
# ============================================================================
# 日期字段原为字符串格式，先转为 datetime，再衍生一个业务特征：
# DAYS_FROM_INCUR_TO_PAY = 支付日期 - 就诊起始日期
# 直觉：理赔处理天数可能与欺诈有关（异常快/异常慢都值得关注）

date_cols = ['INCUR_DATE_FROM', 'INCUR_DATE_TO', 'PAY_DATE', 'RCV_DATE', 'FILE_CLOSE_DATE']
date_cols = [col for col in date_cols if col in df.columns]
for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors='coerce', dayfirst=False)

# 衍生特征：从就诊到支付的天数
if 'PAY_DATE' in df.columns and 'INCUR_DATE_FROM' in df.columns:
    df['DAYS_FROM_INCUR_TO_PAY'] = (df['PAY_DATE'] - df['INCUR_DATE_FROM']).dt.days
else:
    df['DAYS_FROM_INCUR_TO_PAY'] = np.nan

In [ ]:
# Cell 6: 类别列初步处理
# ============================================================================
# 类别字段的问题：大小写不统一、带空格、存在"NAN"等脏标记。
# 统一做 strip + upper 处理，确保同一含义的数据只有一种写法。
# RJ_CODE_LIST 和 CODES 也要清理——它们将在 Cell 11 中用于标签构造。

cat_cols = [
    'PROV_LEVEL', 'MBR_TYPE', 'BEN_HEAD', 'BEN_HEAD_TYPE', 'SCMA_OID_BEN_TYPE',
    'SCMA_OID_CL_LINE_STATUS', 'SCMA_OID_CL_STATUS', 'SCMA_OID_COUNTRY_TREATMENT',
    'KIND_CODE', 'INSUR_INVOICE_IND'
]
cat_cols = [col for col in cat_cols if col in df.columns]
for col in cat_cols:
    df[col] = df[col].astype(str).str.strip().str.upper()
    df[col] = df[col].replace('NAN', np.nan)

# RJ_CODE_LIST 和 CODES 也做同样清理（将用于标签生成）
if 'RJ_CODE_LIST' in df.columns:
    df['RJ_CODE_LIST'] = df['RJ_CODE_LIST'].astype(str).str.strip().str.upper()
    df['RJ_CODE_LIST'] = df['RJ_CODE_LIST'].replace('NAN', np.nan)
if 'CODES' in df.columns:
    df['CODES'] = df['CODES'].astype(str).str.strip().str.upper()
    df['CODES'] = df['CODES'].replace('NAN', np.nan)

In [ ]:
# Cell 7: 其他特殊字段处理
# ============================================================================
# COPAY_PCT（自付比例）、NO_OF_YR（投保年数）、POLICY_CNT（保单数）、
# INVOICE_CNT（发票数）—— 这些字段原始格式可能是字符串，强制转为数值。

if 'COPAY_PCT' in df.columns:
    df['COPAY_PCT'] = pd.to_numeric(df['COPAY_PCT'], errors='coerce')
for col in ['NO_OF_YR', 'POLICY_CNT', 'INVOICE_CNT']:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

In [ ]:
# Cell 8: 缺失值分析（阈值调为 60%）
# ============================================================================
# 缺失值处理策略：
#   - 常量列（方差为0）：直接删除，对分类无贡献
#   - 缺失率 > 60% 的列：删除，填充值不可信
#   - 其余缺失列：在 Cell 9 中做填充处理
#
# FILE_CLOSE_DATE 缺失 63.5%，直接删除；
# CL_THIRD_PARTY_PAY_AMT 为常量列（全是0），无区分能力，删除。

missing_ratio = df.isnull().mean().sort_values(ascending=False)
print("各列缺失比例：")
print(missing_ratio[missing_ratio > 0])

# 常量列：所有值相同（或全为NaN），无法提供区分信息
const_cols = [col for col in df.columns if df[col].nunique(dropna=False) <= 1]
if const_cols:
    print(f"剔除常量列: {const_cols}")
    df.drop(columns=const_cols, inplace=True)

# 缺失率超过 60% 的列：填了也不可信，不如不要
high_missing = missing_ratio[missing_ratio > 0.6].index.tolist()
if high_missing:
    print(f"剔除高缺失率列 (>60%): {high_missing}")
    df.drop(columns=high_missing, inplace=True)

In [ ]:
# Cell 9: 缺失值填充（增强版，确保最终缺失为 0）
# ============================================================================
# 填充策略按数据类型区分：
#   - 数值型：中位数填充（比均值更稳健，不受极端值影响）
#   - 类别型：众数填充（取出现次数最多的值）
#   - 日期型：转换为"距1970-01-01的天数"后用中位数填充
#
# 最后加一道"最终防线"：再跑一遍中位数填充，确保零遗漏。

# 1. 数值型特征：中位数填充
num_features = df.select_dtypes(include=[np.number]).columns.tolist()
imputer_num = SimpleImputer(strategy='median')
df[num_features] = imputer_num.fit_transform(df[num_features])

# 2. 类别型特征：众数填充
cat_features = df.select_dtypes(include=['object']).columns.tolist()
for col in cat_features:
    # 先清理常见的缺失标记字符串
    df[col] = df[col].replace(['NAN', 'NONE', 'NULL', 'NA', ''], np.nan)
    # 若列全为空，直接填 'Unknown'
    if df[col].dropna().empty:
        df[col] = df[col].fillna('Unknown')
    else:
        mode_val = df[col].mode()
        if len(mode_val) > 0:
            df[col] = df[col].fillna(mode_val.iloc[0])
        else:
            df[col] = df[col].fillna('Unknown')

# 3. 日期型特征：转为距1970-01-01的天数（整数），后续可当数值处理
date_cols = df.select_dtypes(include=['datetime64']).columns.tolist()
for col in date_cols:
    df[col] = (df[col] - pd.Timestamp('1970-01-01')).dt.days

# 4. 剔除转换后全为 NaN 的列（如果有）
all_nan_cols = [col for col in df.columns if df[col].isnull().all()]
if all_nan_cols:
    print(f"剔除全缺失列: {all_nan_cols}")
    df.drop(columns=all_nan_cols, inplace=True)

# 5. 最终防线：再次填充所有数值列，确保缺失值为 0
num_cols_now = df.select_dtypes(include=[np.number]).columns.tolist()
if num_cols_now:
    imputer_final = SimpleImputer(strategy='median')
    df[num_cols_now] = imputer_final.fit_transform(df[num_cols_now])

print("缺失值填充完成，当前缺失总数：", df.isnull().sum().sum())

In [ ]:
# Cell 10: 异常值处理 – Winsorize（1% - 99%）
# ============================================================================
# 金额类字段存在极端值（如某笔理赔金额特别大），会影响模型训练。
# 采用 1%-99% 分位数截尾：低于1%的拉到1%，高于99%的压到99%。
# 为什么不直接删除异常值？
#   1. 删掉会减少样本量
#   2. 某些"异常"值本身可能就是欺诈案件，删了就丢信息了
#
# 排除的字段：COPAY_PCT/NO_OF_YR/POLICY_CNT 等取值范围本身较小，
# 以及 DAYS_FROM_INCUR_TO_PAY（衍生特征）和分箱列。

numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
exclude = ['COPAY_PCT', 'NO_OF_YR', 'POLICY_CNT', 'INVOICE_CNT', 'DAYS_FROM_INCUR_TO_PAY']
numeric_cols = [c for c in numeric_cols if c not in exclude and df[c].nunique() > 10]

for col in numeric_cols:
    lower = df[col].quantile(0.01)
    upper = df[col].quantile(0.99)
    df[col] = df[col].clip(lower, upper)
print("异常值处理（1% - 99% 截尾）完成")

In [ ]:
# Cell 11: 【核心】标签构造 + 数据泄露防范
# ============================================================================
# 这是整个预处理流程中最关键的一步——定义"什么是欺诈"。
#
# 欺诈定义（业务逻辑）：
#   条件1：拒赔码包含 R530 或 T180（系统中标识欺诈相关拒赔的专用代码）
#   条件2：实际支付金额为零（理赔未被支付）
#   同时满足两个条件 → 标记为欺诈（FRAUD=1）
#
# 辅助特征构造：
#   HAS_R530  = 拒赔码列表中是否包含 R530
#   HAS_T180  = 拒赔码列表中是否包含 T180
#   IS_ZERO_PAY = 是否零支付（PAY_AMT_USD==0 或 REJECTED_AMT<0）
#
# 【数据泄露防范】标签构造完成后，立即删除 RJ_CODE_LIST 和 CODES。
# 原因：标签就是从这两个字段提取的，如果不删，模型训练时会"看到答案"，
# 导致训练集表现极好但真实数据上完全不work。

# 构造辅助特征：是否含有 R530 / T180 等拒赔码
if 'RJ_CODE_LIST' in df.columns:
    df['HAS_R530'] = df['RJ_CODE_LIST'].str.contains('R530', na=False).astype(int)
    df['HAS_T180'] = df['RJ_CODE_LIST'].str.contains('T180', na=False).astype(int)
else:
    df['HAS_R530'] = 0
    df['HAS_T180'] = 0

# 定义"零支付"案件
# REJECTED_AMT < 0 表示拒赔金额（负数记账），与零支付逻辑一致
if 'PAY_AMT_USD' in df.columns and 'REJECTED_AMT' in df.columns:
    df['IS_ZERO_PAY'] = ((df['PAY_AMT_USD'] == 0) |
                        (df['REJECTED_AMT'].notna() & (df['REJECTED_AMT'] < 0))).astype(int)
elif 'PAY_AMT_USD' in df.columns:
    df['IS_ZERO_PAY'] = (df['PAY_AMT_USD'] == 0).astype(int)
else:
    df['IS_ZERO_PAY'] = 0

# 构造欺诈标签：含有 R530 或 T180 且为零支付
df['FRAUD'] = ((df['HAS_R530'] == 1) | (df['HAS_T180'] == 1)) & (df['IS_ZERO_PAY'] == 1)
df['FRAUD'] = df['FRAUD'].astype(int)
print("欺诈标签分布：\n", df['FRAUD'].value_counts())

# 关键：立即删除原始泄露列 RJ_CODE_LIST 和 CODES
leakage_cols = ['RJ_CODE_LIST', 'CODES']
leakage_cols = [c for c in leakage_cols if c in df.columns]
if leakage_cols:
    df.drop(columns=leakage_cols, inplace=True)
    print(f"已删除泄露特征: {leakage_cols}")

In [ ]:
# Cell 12: 分箱处理
# ============================================================================
# 金额字段（SUB_AMT、PAY_AMT_USD）是连续数值，但对欺诈检测来说，
# 我们更关心"金额处于什么档次"而非具体数字。
# 使用 5 分位分箱，把金额变成 0-4 的离散等级。
#
# 注意分箱时机：在对数变换之前做，因为分箱基于原始金额分布来划分。
# 分箱后删除原始连续列，避免后续被对数变换或标准化干扰。

discretize_cols = ['SUB_AMT', 'PAY_AMT_USD']
discretize_cols = [c for c in discretize_cols if c in df.columns]

if discretize_cols:
    est = KBinsDiscretizer(n_bins=5, encode='ordinal', strategy='quantile')
    binned_data = est.fit_transform(df[discretize_cols]).astype(int)
    binned_cols = [f"{col}_BIN" for col in discretize_cols]
    df.drop(columns=discretize_cols, inplace=True)
    df[binned_cols] = binned_data
    print(f"分箱完成，生成新特征: {binned_cols}")
else:
    print("无合适的分箱特征")

In [ ]:
# Cell 13: 对数变换
# ============================================================================
# 很多金额字段分布严重右偏（大部分值很小，少数极大），会影响模型性能。
# 对偏度 > 1 的连续特征做 log1p 变换（即 log(1+x)），拉近分布对称性。
#
# 如果特征存在零值或负值，先平移（减最小值+1）保证对数运算有效。
# 排除标签列、辅助特征、百分比/计数字段和分箱列。

candidate_numeric = [c for c in df.select_dtypes(include=[np.number]).columns
                     if c not in ['FRAUD', 'HAS_R530', 'HAS_T180', 'IS_ZERO_PAY',
                                  'COPAY_PCT', 'NO_OF_YR', 'POLICY_CNT', 'INVOICE_CNT']
                     and not c.endswith('_BIN')]
skewness = df[candidate_numeric].skew().abs()
skewed_cols = skewness[skewness > 1].index.tolist()
print("偏度 > 1 的特征（将进行log变换）：", skewed_cols)

for col in skewed_cols:
    min_val = df[col].min()
    if min_val <= 0:
        # 平移至正数域，确保 log1p 有效
        df[col] = df[col] - min_val + 1
    df[col] = np.log1p(df[col])
    print(f"已将 {col} 进行对数变换")

In [ ]:
# Cell 14: 标准化（StandardScaler）
# ============================================================================
# 不同字段量纲差异大（金额可能是几十万，投保年数可能是个位数），
# 不标准化的话，量纲大的字段会主导模型训练。
# StandardScaler 将所有数值特征转为均值=0、标准差=1 的分布。
#
# 排除：标签列 FRAUD、辅助特征 HAS_R530/HAS_T180/IS_ZERO_PAY（0/1 标记）、
#       分箱列（已是离散等级，标准化无意义）。

exclude_scaling = ['FRAUD', 'HAS_R530', 'HAS_T180', 'IS_ZERO_PAY']
bin_like = [c for c in df.columns if c.endswith('_BIN')]
scaler = StandardScaler()
numeric_to_scale = [c for c in df.select_dtypes(include=[np.number]).columns
                    if c not in exclude_scaling + bin_like]

df_scaled = df.copy()
df_scaled[numeric_to_scale] = scaler.fit_transform(df[numeric_to_scale])
print("标准化完成，缩放特征：", numeric_to_scale)

In [ ]:
# Cell 15: 特征编码（Label Encoding）
# ============================================================================
# 剩余的类别字段（诊断编码、保障计划描述、险种编码等）需要变成数值才能建模。
# Label Encoding 给每个字符串类别分配一个整数。
#
# 注意：Label Encoding 会引入"顺序"关系（0 < 1 < 2），但实际上类别之间
# 可能没有大小关系。后续特征选择阶段使用随机森林（非线性模型）来缓解此问题。

cat_features = df_scaled.select_dtypes(include=['object']).columns.tolist()
print("待编码的类别特征：", cat_features)

label_encoders = {}
for col in cat_features:
    le = LabelEncoder()
    df_scaled[col] = le.fit_transform(df_scaled[col].astype(str))
    label_encoders[col] = le
print("类别特征编码完成（Label Encoding）")

In [ ]:
# Cell 16: 最终验证
# ============================================================================
# 预处理完成后的检查点：确认数据形状、类型、缺失值等是否符合预期。
# 期望：76911 行 × 47 列，缺失值为 0，全部为数值类型。

print("预处理后数据形状：", df_scaled.shape)
print("数据类型分布：\n", df_scaled.dtypes.value_counts())
print("缺失值总数：", df_scaled.isnull().sum().sum())
print("\n数值特征统计：")
display(df_scaled.describe().T.head(10))

In [ ]:
# Cell 17: 保存预处理后的完整数据集
# ============================================================================
# 保存包含所有 47 列的预处理数据，作为中间产物。
# 使用 utf-8-sig 编码确保中文环境下的兼容性。

output_path = 'preprocessed_data.csv'
df_scaled.to_csv(output_path, index=False, encoding='utf-8-sig')
print(f"预处理完毕，数据已保存至 {output_path}")

In [ ]:
# Cell 18: 分离特征矩阵 X 和目标向量 y，剔除 ID 列
# ============================================================================
# ID 列（理赔号、会员号、保单号等）是标识符，不具备预测能力，
# 留在特征里反而会增加噪声和过拟合风险，直接剔除。
# 剔除后剩下 37 个候选特征，进入特征选择阶段。

id_cols = [
    'CL_NO', 'CL_LINE_NO', 'MBR_NO', 'POCY_NO', 'POPL_OID', 'POHO_NO',
    'PLAN_OID', 'PROV_CODE', 'CLSH_HOSP_CODE'
]
id_cols = [c for c in id_cols if c in df_scaled.columns]

y = df_scaled['FRAUD']
X = df_scaled.drop(columns=id_cols + ['FRAUD'])
print(f"原始特征数量: {X.shape[1]}")
print("已剔除的ID列:", id_cols)

In [ ]:
# Cell 19: 过滤法特征选择 — 低方差 + 单变量相关性
# ============================================================================
# 过滤法是"不依赖模型、纯基于数据统计特性"的筛选方式。
#
# 第一步：低方差过滤（VarianceThreshold=0.01）
#   直觉：特征在所有样本上取值几乎不变 → 无法区分欺诈/非欺诈 → 删除
#   37 个特征去掉 2 个，剩 35 个。
#
# 第二步：单变量相关性过滤（SelectKBest, F检验）
#   计算每个特征与标签的 F 统计量，选出相关性最强的 20 个。
#   35 个去掉 15 个，剩 20 个。

from sklearn.feature_selection import VarianceThreshold, SelectKBest, f_classif

# 低方差过滤
sel_var = VarianceThreshold(threshold=0.01)
X_var = sel_var.fit_transform(X)
mask_var = sel_var.get_support()
kept_features_var = X.columns[mask_var]
print(f"低方差过滤后保留特征数: {len(kept_features_var)}")

# 单变量相关性过滤
k_best = min(20, len(kept_features_var))
selector = SelectKBest(score_func=f_classif, k=k_best)
X_filtered = selector.fit_transform(X_var, y)
mask = selector.get_support()
final_filter_features = kept_features_var[mask]
print("过滤法最终保留的特征：")
print(final_filter_features.tolist())

X_final = pd.DataFrame(X_filtered, columns=final_filter_features)

In [ ]:
# Cell 20: 嵌入法特征选择 — 基于随机森林的特征重要性
# ============================================================================
# 嵌入法是"训练一个模型，让它告诉我们哪些特征重要"的筛选方式。
# 使用 100 棵树的随机森林，基于基尼不纯度减少量计算特征重要性。
#
# 结果：IS_ZERO_PAY(0.28) + REJECTED_AMT(0.21) + COPAY_PCT(0.18) + HAS_R530(0.14)
#       前四个特征就占了超过 80% 的重要性，与标签定义高度吻合。
#
# SelectFromModel 以中位数为阈值：只保留重要性高于中位数的特征。
# 20 个 → 10 个。

from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectFromModel

rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_final, y)

importances = pd.Series(rf.feature_importances_, index=X_final.columns).sort_values(ascending=False)
print("特征重要性 Top 10：")
print(importances.head(10))

sel_embedded = SelectFromModel(rf, threshold='median')
X_embedded = sel_embedded.fit_transform(X_final, y)
embedded_features = X_final.columns[sel_embedded.get_support()]
print(f"\n嵌入法保留的特征数: {len(embedded_features)}")
print("保留的特征：", embedded_features.tolist())

In [ ]:
# Cell 21: 包装法特征选择（RFE）
# ============================================================================
# 包装法的想法："用模型实际跑一遍，看去掉哪个特征表现下降最多"。
# 使用 RFE（递归特征消除）逐步剔除最不重要的特征。
#
# 但嵌入法之后只剩 10 个特征，已经达到目标数量，跳过包装法。
# 如果嵌入法保留的特征多于 10 个，才会执行 RFE 进一步筛选。
#
# 基模型选择 RandomForestClassifier 而非线性模型，
# 避免 Label Encoding 引入的序数假设对线性模型造成误导。

from sklearn.feature_selection import RFE

if X_embedded.shape[1] > 10:
    rfe_estimator = RandomForestClassifier(n_estimators=50, random_state=42, n_jobs=-1)
    rfe = RFE(rfe_estimator, n_features_to_select=10, step=1)
    X_rfe = rfe.fit_transform(X_embedded, y)
    rfe_features = embedded_features[rfe.get_support()]
    print("包装法最终保留的特征：")
    print(rfe_features.tolist())
    X_selected = pd.DataFrame(X_rfe, columns=rfe_features)
else:
    X_selected = pd.DataFrame(X_embedded, columns=embedded_features)
    print("特征已精简，跳过包装法。")

In [ ]:
# Cell 22: 保存最终建模数据
# ============================================================================
# 将筛选后的 10 个特征 + FRAUD 标签合并，保存为最终建模数据集。
# 下游任务拿到 data_for_modeling.csv 就可以直接做模型训练，
# 无需再做数据清洗、特征编码、缺失值处理等操作。
#
# 最终输出：76,911 行 × 11 列（10 特征 + 1 标签）
# 最终特征：APP_AMT, REJECTED_AMT, COPAY_PCT, PROV_LEVEL, POCY_PLAN_DESC,
#           KIND_CODE, CWF_AMT_DAY, HAS_R530, HAS_T180, IS_ZERO_PAY

final_df = pd.concat([X_selected, y.rename('FRAUD')], axis=1)
final_df.to_csv('data_for_modeling.csv', index=False, encoding='utf-8-sig')
print(f"最终建模数据集形状: {final_df.shape}")
print("文件已保存为 data_for_modeling.csv")